# BRAMASTRA Gandiva — 100M TPU zero-update preflight

This is a **separate TPU notebook**; it does not alter or replace `bramastra_k8.ipynb` or the two-T4 GPU campaign. Select Kaggle's **8-core TPU v3-8 / TPU VM v3-8** accelerator (use the matching label shown in your session settings) and enable Internet so the first cell can clone the pushed `Gandiva` branch. If no prepared K8 bundle is attached, the notebook builds the registered deterministic bundle in `/kaggle/working`.

The run checks all eight TPU replicas, the exact 100,334,720-parameter configuration, synchronized initial weights, and one real B-arm multi-objective forward/backward window per replica. It **never calls the optimizer update boundary**. It is a preflight, not model training or evidence of AGI/task competence. It does not qualify optimizer-state memory, a committed update, checkpoint/resume, sustained throughput, or a full training campaign. A pass tells us the TPU backward path executed; it does not mean the 100M campaign is ready.

Use a fresh Kaggle session and run all cells in order. The final cell creates a ZIP and SHA-256 receipt under `/kaggle/working`; download the ZIP from Kaggle's Output panel. Kaggle's [TPU documentation](https://www.kaggle.com/docs/tpu) and [Notebook documentation](https://www.kaggle.com/docs/notebooks) describe the available TPU and session constraints.


In [ ]:
import importlib.util
import os
from pathlib import Path
import re
import subprocess
import sys

WORKING = Path(os.environ.get('BRAMASTRA_WORKING', '/kaggle/working'))
INPUT = Path(os.environ.get('BRAMASTRA_INPUT', '/kaggle/input'))
WORKING.mkdir(parents=True, exist_ok=True)
GIT_URL = os.environ.get(
    'BRAMASTRA_GIT_URL',
    'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git')
GIT_REF = os.environ.get('BRAMASTRA_GIT_REF', 'Gandiva')
AUTO_SOURCE = WORKING / 'bramastra-gandiva-tpu-source'

if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._/-]{0,127}', GIT_REF):
    raise RuntimeError('BRAMASTRA_GIT_REF contains unsafe characters.')

def is_source_tree(path):
    path = Path(path)
    return (path / 'pyproject.toml').is_file() and (path / 'bramastra_lab').is_dir()

configured = os.environ.get('BRAMASTRA_REPO')
if configured:
    REPO = Path(configured).expanduser().resolve()
    if not is_source_tree(REPO):
        raise RuntimeError(f'BRAMASTRA_REPO is not a source tree: {REPO}')
elif AUTO_SOURCE.exists():
    if not is_source_tree(AUTO_SOURCE):
        raise RuntimeError(
            f'{AUTO_SOURCE} exists but is not a BRAMASTRA source tree; '
            'the notebook will not delete or replace it.')
    status = subprocess.run(
        ['git', '-C', str(AUTO_SOURCE), 'status', '--porcelain'],
        capture_output=True, text=True, check=True)
    if status.stdout.strip():
        raise RuntimeError(
            'The existing Gandiva clone has local changes. Preserve or move it, '
            'then start a fresh Kaggle session; this notebook will not reset it.')
    subprocess.run(['git', '-C', str(AUTO_SOURCE), 'pull', '--ff-only',
                    'origin', GIT_REF], check=True)
    REPO = AUTO_SOURCE.resolve()
else:
    clone = subprocess.run(
        ['git', 'clone', '--depth', '1', '--single-branch', '--branch',
         GIT_REF, GIT_URL, str(AUTO_SOURCE)], capture_output=True, text=True)
    if clone.returncode != 0 or not is_source_tree(AUTO_SOURCE):
        detail = (clone.stderr or clone.stdout).strip()[-1200:]
        raise RuntimeError(
            'Automatic Gandiva clone failed. Enable Internet in Kaggle Session '
            'options, select TPU v3-8, restart the session and retry. Git detail: '
            + detail)
    REPO = AUTO_SOURCE.resolve()

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
missing = [name for name in ('torch', 'torch_xla', 'numpy')
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        'Kaggle TPU runtime is missing required modules: ' + ', '.join(missing)
        + '. Select TPU v3-8 and restart the session; do not pip-upgrade '
        'torch or torch_xla inside this preflight.')
revision = subprocess.run(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'],
    capture_output=True, text=True, check=True).stdout.strip()
print('source:', REPO)
print('branch:', GIT_REF)
print('source revision:', revision)


In [ ]:
from datetime import datetime, timezone
import json
import subprocess

from bramastra_lab.research.campaigns.kaggle_env import (
    discover_bundle, is_k8_bundle)
from bramastra_lab.research.data.k8_bundle import validate_bundle

BUNDLE_PATH, _checked = discover_bundle()
if BUNDLE_PATH is None:
    BUNDLE_PATH = WORKING / 'bramastra-tpu-100m-inputs'
    if BUNDLE_PATH.exists() and not is_k8_bundle(BUNDLE_PATH):
        raise RuntimeError(
            f'{BUNDLE_PATH} exists but is not a valid K8 bundle; '
            'the notebook will not delete or overwrite it.')
    if not is_k8_bundle(BUNDLE_PATH):
        prepare = [
            sys.executable, '-m', 'bramastra_lab.research.campaigns.k8',
            'prepare', '--out', str(BUNDLE_PATH),
            '--training-mechanisms', '4096',
            '--controller-mechanisms', '256',
            '--development-mechanisms', '256',
            '--confirmation-mechanisms', '128',
            '--tool-mechanisms', '4096', '--tool-heldout', '256',
            '--meta-train', '24', '--meta-validate', '6', '--meta-confirm', '6']
        subprocess.run(prepare, cwd=REPO, check=True)

BUNDLE_PATH = Path(BUNDLE_PATH).resolve()
validation = validate_bundle(str(BUNDLE_PATH))
if not validation.get('valid'):
    raise RuntimeError(
        'K8 data bundle validation failed: '
        + json.dumps(validation.get('issues', [])[:12], sort_keys=True))
BUNDLE_DIR = str(BUNDLE_PATH)
SEED = 1701
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
REPORT_DIR = WORKING / f'bramastra-tpu-100m-preflight-{RUN_ID}'
if REPORT_DIR.exists() and any(REPORT_DIR.iterdir()):
    raise RuntimeError(f'Run directory already contains evidence: {REPORT_DIR}')
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print('bundle:', BUNDLE_DIR)
print('bundle identity:', validation.get('identity'))
print('training rows:', validation.get('counts', {}).get('training'))
print('report directory:', REPORT_DIR)
print('seed:', SEED)


In [ ]:
import json
import traceback

from bramastra_lab.research.campaigns.tpu_100m import run_tpu_100m_preflight

PREFLIGHT_REPORT = None
PREFLIGHT_FAILURE = None
try:
    PREFLIGHT_REPORT = run_tpu_100m_preflight(
        data_dir=BUNDLE_DIR, report_dir=str(REPORT_DIR), seed=SEED)
    print(json.dumps(PREFLIGHT_REPORT, sort_keys=True, indent=2))
except Exception as exc:
    PREFLIGHT_FAILURE = {
        'status': 'PREFLIGHT_FAILED',
        'error': f'{type(exc).__name__}: {exc}',
        'training_started': False,
    }
    print(json.dumps(PREFLIGHT_FAILURE, sort_keys=True, indent=2))
    traceback.print_exc()


In [ ]:
import hashlib
import json
import zipfile
from IPython.display import FileLink, display

if not REPORT_DIR.is_dir():
    raise RuntimeError(f'Preflight report directory is missing: {REPORT_DIR}')
archive = WORKING / f'{REPORT_DIR.name}-artifacts.zip'
partial = WORKING / f'.{REPORT_DIR.name}-artifacts.partial.zip'
receipt = archive.with_suffix('.json')
if archive.exists() or receipt.exists() or partial.exists():
    raise RuntimeError('Artifact output already exists; use a fresh Kaggle session.')
with zipfile.ZipFile(partial, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(REPORT_DIR.rglob('*')):
        if path.is_file() and not path.is_symlink():
            bundle.write(path, arcname=Path(REPORT_DIR.name) / path.relative_to(REPORT_DIR))
with zipfile.ZipFile(partial) as bundle:
    bad_member = bundle.testzip()
if bad_member is not None:
    raise RuntimeError(f'Artifact ZIP failed integrity check at {bad_member}')
digest = hashlib.sha256(partial.read_bytes()).hexdigest()
partial.replace(archive)
receipt.write_text(json.dumps({
    'schema': 'bramastra-tpu-100m-preflight-archive/v1',
    'status': (PREFLIGHT_REPORT or PREFLIGHT_FAILURE or {}).get('status', 'UNKNOWN'),
    'archive': archive.name,
    'sha256': digest,
    'bytes': archive.stat().st_size,
    'report_directory': REPORT_DIR.name,
}, sort_keys=True, indent=2) + '\n', encoding='utf-8')
print('preflight artifact:', archive)
print('sha256:', digest)
display(FileLink(str(archive)))
